# TN4 — Test cuối trên GHIJ, DS-TCN 64 kênh tầm nhìn 61

## Đây là số công bố

Train trên **đủ 8 người ABCDEFKL**, chấm **một lần** trên **GHIJ** — 537 buổi
ghi của 4 người chưa từng xuất hiện ở bất kỳ bước chọn cấu hình nào.

Khác với `run_cv.py`: ở đó model chỉ thấy 6 người mỗi fold và điểm dùng để
**chọn**. Ở đây model thấy cả 8 người và điểm dùng để **báo cáo**. Theo đúng
`docs/PROTOCOL.md` mục 6.

## Cấu hình — đã chốt xong, không chọn gì thêm ở đây

| | |
|---|---|
| model | `ds_tcn --channels 64 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **37.081** |
| tầm nhìn | 61 |
| loss | `mse_pearson --alpha 0,6` |
| điểm dev của cấu hình này | **0,780028** (TN3, 4 fold, 1 seed) |
| cùng kiến trúc, MSE thuần | 0,760878 |

`alpha 0,6` chọn từ **TN3 trên tập dev**, là mức cao nhất của chính bề
rộng kênh này. GHIJ không tham gia vào lựa chọn đó.

**Một giới hạn phải ghi khi báo cáo.** TN3 cho thấy thứ hạng giữa các alpha
không chuyển được giữa hai bề rộng kênh — tốp 3 của c64 (0,6 · 0,5 · 0,4) và
của c192 (0,2 · 0,3 · 0,0) không giao nhau mức nào, và biên độ dao động bên
trong mỗi cột cùng cỡ với dao động giữa các seed. Nên `alpha 0,6` là
**mức tốt nhất đo được trên dev**, không phải mức tối ưu đã chứng minh.

Điều TN3 chứng minh được là **20/20 mức alpha đều hơn MSE thuần**, ở cả hai cấu
hình. Đó mới là phát biểu đem vào luận văn.

## Ba seed

Ba seed cho ra `seed_std` của chính con số công bố — thứ trả lời được câu hỏi
"nhỡ ăn may thì sao". Mỗi lần chạy khoảng **25 phút**, tổng khoảng
**1,3 giờ**.

`run_final_test.py` **tự nén và chép sang Drive sau mỗi lần chạy**, tên tệp nén
có cả cấu hình lẫn seed nên không đè nhau. Không cần ô lưu riêng.

Dừng giữa chừng cũng được: mở lại, chạy ô khôi phục ở mục 1 rồi bấm tiếp seed
còn thiếu. Seed đã xong sẽ in `đã có kết quả macro ... — không chạy lại.`

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn.

In [2]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : bf1ac10
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive. Test cuối đọc `windows/final_train/`, cắt gộp cả 8 người theo đúng thứ tự MobiVital.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Khôi phục các seed đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Nó gộp `runs/*/summary.csv` vào `runs/summary.csv` — chỗ `run_final_test.py` tra để biết seed nào đã xong.

In [4]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn4_*c64_*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

khôi phục 0 dòng vào runs/summary.csv


## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **37.081**.

In [5]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   37081

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 3, 4 khối -> tầm nhìn 61, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 61/200 mẫu gần nhất, mất 70% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT CẢ

## 3. Ba seed

Mỗi ô train lại từ đầu trên ABCDEFKL rồi chấm GHIJ.

**seed 0**

In [6]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 0

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k3_n4_none_do0.2_dpel_mse_pearson_a0.6_corr0.9_seed0
thiết bị Tesla T4

292708 cửa sổ train
37081 tham số

epoch  0  mse 0.06473  pearson 0.4976   0.6 phút
epoch  1  mse 0.04013  pearson 0.5535   1.2 phút
epoch  2  mse 0.03738  pearson 0.5656   1.7 phút
epoch  3  mse 0.03578  pearson 0.5739   2.3 phút
epoch  4  mse 0.03420  pearson 0.5799   2.9 phút
epoch  5  mse 0.03292  pearson 0.5849   3.5 phút
epoch  6  mse 0.03191  pearson 0.5886   4.0 phút
epoch  7  mse 0.03110  pearson 0.5913   4.6 phút
epoch  8  mse 0.03058  pearson 0.5940   5.2 phút
epoch  9  mse 0.03006  pearson 0.5964   5.8 phút
epoch 10  mse 0.02974  pearson 0.5977   6.3 phút
epoch 11  mse 0.02940  pearson 0.5996   6.9 phút
epoch 12  mse 0.02905  pearson 0.6010   7.5 phút
epoch 13  mse 0.02891  pearson 0.6025   8.0 phút
epoch 14  mse 0.02861  pearson 0.6042   8.6 phút
epoch 15  mse 0.02833  pearson 0.6043   9.2 phút
epoch 16  mse 0.02824  pearson 0.6063   9.8 phút
epoch 17  

**seed 1**

In [7]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 1

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k3_n4_none_do0.2_dpel_mse_pearson_a0.6_corr0.9_seed1
thiết bị Tesla T4

292708 cửa sổ train
37081 tham số

epoch  0  mse 0.06724  pearson 0.5012   0.6 phút
epoch  1  mse 0.03790  pearson 0.5581   1.2 phút
epoch  2  mse 0.03471  pearson 0.5687   1.8 phút
epoch  3  mse 0.03283  pearson 0.5755   2.3 phút
epoch  4  mse 0.03151  pearson 0.5813   2.9 phút
epoch  5  mse 0.03044  pearson 0.5868   3.5 phút
epoch  6  mse 0.02978  pearson 0.5906   4.0 phút
epoch  7  mse 0.02917  pearson 0.5942   4.6 phút
epoch  8  mse 0.02857  pearson 0.5979   5.2 phút
epoch  9  mse 0.02819  pearson 0.6007   5.7 phút
epoch 10  mse 0.02801  pearson 0.6024   6.3 phút
epoch 11  mse 0.02775  pearson 0.6038   6.9 phút
epoch 12  mse 0.02762  pearson 0.6050   7.4 phút
epoch 13  mse 0.02741  pearson 0.6062   8.0 phút
epoch 14  mse 0.02732  pearson 0.6071   8.6 phút
epoch 15  mse 0.02716  pearson 0.6080   9.1 phút
epoch 16  mse 0.02717  pearson 0.6089   9.7 phút
epoch 17  

**seed 2**

In [8]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 2

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k3_n4_none_do0.2_dpel_mse_pearson_a0.6_corr0.9_seed2
thiết bị Tesla T4

292708 cửa sổ train
37081 tham số

epoch  0  mse 0.06916  pearson 0.4920   0.6 phút
epoch  1  mse 0.03970  pearson 0.5549   1.1 phút
epoch  2  mse 0.03617  pearson 0.5662   1.7 phút
epoch  3  mse 0.03337  pearson 0.5751   2.3 phút
epoch  4  mse 0.03157  pearson 0.5813   2.8 phút
epoch  5  mse 0.03034  pearson 0.5870   3.4 phút
epoch  6  mse 0.02954  pearson 0.5915   4.0 phút
epoch  7  mse 0.02904  pearson 0.5954   4.5 phút
epoch  8  mse 0.02871  pearson 0.5966   5.1 phút
epoch  9  mse 0.02855  pearson 0.5986   5.6 phút
epoch 10  mse 0.02836  pearson 0.6001   6.2 phút
epoch 11  mse 0.02815  pearson 0.6019   6.8 phút
epoch 12  mse 0.02818  pearson 0.6025   7.3 phút
epoch 13  mse 0.02797  pearson 0.6036   7.9 phút
epoch 14  mse 0.02780  pearson 0.6043   8.5 phút
epoch 15  mse 0.02779  pearson 0.6054   9.0 phút
epoch 16  mse 0.02763  pearson 0.6066   9.6 phút
epoch 17  

## 4. Kết quả

`score_macro` là **Pearson macro theo người** trên GHIJ — trung bình từng người, rồi trung bình bốn người.

In [9]:
import csv, os
import statistics as st

# run_final_test.py ghi mỗi lần chạy MỘT dòng, không có dòng TONG như run_cv.py.
rows = []
if os.path.exists("runs/tn4/summary.csv"):
    rows = [r for r in csv.DictReader(open("runs/tn4/summary.csv"))
            if "_c64_" in r["run_id"]]

diem = {}
for r in rows:
    diem[int(r["seed"])] = float(r["score_macro"])

for s in sorted(diem):
    print("  seed", s, " ", round(diem[s], 6))
if len(diem) >= 2:
    v = list(diem.values())
    print("  " + "-" * 34)
    print("  trung bình", round(st.mean(v), 6))
    print("  seed_std  ", round(st.stdev(v), 6))
print()
print("  MỐC ĐỐI CHIẾU trên GHIJ")
for ten, d in (("MobiVital công bố", "0,819"),
               ("LSTM-352, 1.502.713 tham số", "0,810302"),
               ("LSTM-67", "0,801683"),
               ("DS-TCN-64 bản đầu", "0,795783")):
    print("   ", ten.ljust(30), d)

  seed 0   0.786906
  seed 1   0.817115
  seed 2   0.806751
  ----------------------------------
  trung bình 0.80359
  seed_std   0.01535

  MỐC ĐỐI CHIẾU trên GHIJ
    MobiVital công bố              0,819
    LSTM-352, 1.502.713 tham số    0,810302
    LSTM-67                        0,801683
    DS-TCN-64 bản đầu              0,795783


## 5. Ngắt phiên

In [10]:
from google.colab import runtime
runtime.unassign()